# Basic analysis of the opsim database

- **author :** Sylvie Dagoret-Campagne
- **affliliation :** IJCLab/IN2P3/CNRS
- **creation date :** 2026-07-23
- **last date :** 2026-07-24 : add info on Opsims

## Requirements
- Install :
https://github.com/LSSTDESC/OpSimSummaryV2

In [ ]:
import opsimsummaryv2 as op
import os
import sys
import logging
import re

## 1) Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging implemented !")


## 2) Db Configuration

In [ ]:
# Notebook input

# ── Notebook tag ──────────────────────────────────────────────────────
NB_TAG = "ExampleExploreDB_01"
DIR_DATA_IN = "/Users/dagoret/DATA/OpSim"

# ── Output figures ────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

### List of opsim simulations
See the post at : https://community.lsst.org/t/release-of-v5-3-simulations/12032

```
he start of the LSST is fast approaching, and there has been significant interest in new simulations that would include more up-to-date knowledge and expectations of the system’s likely performance in year one and onwards. We have been working hard to better understand our system, we are making progress, and we appreciate your patience while this work is unfolding. System improvements will continue throughout the LSST, and maintenance and engineering time to accomplish them is being built into the schedule now.

The v5.3 simulations described and linked below represent our current best estimates for the system on-sky uptime, slew performance, and engineering time needs, which are still evolving. We will update simulations as we gain more reliable knowledge. The simulation’s start date is consistent with the start date for LSST (in the next months), but should not be taken as a forecast of or commitment to the official start date of the survey. The start date of the survey will be proposed by Rubin’s Start of LSST Board to the Rubin Observatory Director, who will make the final decision after consulting with the Rubin Management Board and funding agencies.

The biggest changes are in the amount of time available for on-sky for observations. The addition of more maintenance time in every year (46 weeks total instead of 20), additional engineering test time (~1 night every month, as well as two half-nights a week for the first six months), and a more realistic fault time per night result in a reduction of the overall number of visits on-sky of about 10% compared to v5.0 simulations. This 10% reduction is seen in both year 1 and in all years (while previous simulations, post v3.6, already included significant additional downtime within year 1).

The baseline v5.3.0 simulation 43 represents our best estimate of an update of the baseline v5.0 observing strategy. This simulation follows the Phase 3 SCOC recommendations PSTN-056 22 and contains the same footprint and rolling cadence as v5.0, similar DDF and ToO programs, no snaps (the Observatory has consistently taken recent observations without snaps - that is each “visit” is a single exposure of 30 (38) seconds in grizy (u)).

Dithering between visits in a night in the DDFs is added to dithering between nights.

A notable change in v5.3 is the addition of a template tier of surveys in Y1, which aims to obtain “template-quality images” (i.e., images with low atmospheric seeing) in pairs. The addition of this tier has arisen from a better understanding of the (still evolving) requirements for creating template images for difference image analysis. Pairs enable Solar System science before the creation of templates and provide colors for transients, even if discovery is not possible without templates. This tier pushes the acquisition of the needed minimum number of template input images to a higher priority than they would have otherwise.

Additional simulations in the v5.3 release include:

    comp_survey_v5.3.0_10yrs.db 8 implements the v5.3 survey strategy with the v5.0 available on-sky time model. This enables a 1-1 comparison with previous surveys.
    Simulations varying the rolling cadence cycles. If templates are not collected over the full footprint at the end of year 1, delaying the start of rolling cadence until year 2 allows us to extend the template-tier survey throughout year 2. This affects the timing of “uniform data releases” (the goal of the “uniform rolling cadence” developed in v5.0). In the baseline, these “uniform” data releases are at Y1, Y4, Y7, and Y10. Delaying rolling puts them at Y1, Y2, Y10 – and either Y7 or Y5. The roll_mash_v5.3.0_10yrs.db places this at Y7, while the roll_u5_v5.3.0_10yrs.db simulation places it at Y5.
    faster_templates_v5.3.0_10yrs.db 2 is a potential variation, which has not yet been verified as a valid option for use on-sky and with data management pipelines, where the template tier visits are reduced in exposure time from 30s (38s) in grizy (u) to 20s (25s). However, this option would require some consideration of calibration and data management concerns, which have not yet been validated. It also reduces the on-sky duty cycle and brings the u-band visits into a read-noise dominated regime.
    ddf_sd_v5.3.0_10yrs.db, looks at increasing the short (majority of) DDF sequences by a visit or two in the bluer bands. This facilitates AGN monitoring science. This simulation pushes the DDF fraction of time over 8% (recall the SCOC recommended containing the program within 7%, PSTN-055 and PSTN-056 22). With greater up-time on-sky, either through better weather than in our model or better observatory performance, this could still meet or get close to the SCOC recommendations.
    A simulation evaluating the impact of moving a significant fraction of u and g band visits into years 1-4 in the northern section of our footprint, for the purpose of building coadded depth to support DESI target choices, is evaluated in the desi_3040_v5.3.0_10yrs.db 4 simulation. This has a significant impact on the cadence in the northern region (very little to no u and g band images in the region in the years immediately following Y4), as well as the uniformity at years 1 through 4, although uniformity evens out by the end of the survey.
    A simulation running LSST for 11 years is considered in baseline_v5.3.0_11years.db 3 to assess how science would recover from the impact of increased engineering time models with additional survey time. We have advocated in the past and continue to advocate to NSF and DOE the need to reach all the LSST science goals. We include a (not approved) year 11 in our plans so that costs may be considered with the (simulated) benefits.
    We also include our usual set of “weather variations” – where the baseline survey is run against different years of our weather records. These help us understand uncertainty in metrics that may be coming from weather only.

```

- The set of v5.3 simulation databases is available for download here Index of https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/
- A high-level summary of metric comparisons is available here https://github.com/lsst-pst/survey_strategy/blob/main/fbs_5.3/v5.3_Update.ipynb
- The full MAF metric outputs are available as always at https://usdf-maf.slac.stanford.edu/ 

In [ ]:
# Opsim Rubin-LSST cadence simulations
list_opsims = ["baseline_v5.3.0_10yrs.db","faster_templates_v5.3.0_10yrs.db","ddf_one_less_v5.3.2_10yrs.db","roll_mash_v5.3.0_10yrs.db",
               "roll_u5_v5.3.0_10yrs.db","ddf_sd_v5.3.0_10yrs.db","desi_3040_v5.3.0_10yrs.db","baseline_v5.3.0_11yrs.db"]

In [ ]:
index_sim = 0
filename_in = list_opsims[index_sim]
opsim_path = os.path.join(DIR_DATA_IN,filename_in)

In [ ]:
log.info("Selected Opsim candence simulation : %s", filename_in)
tagname = re.match(r"(.+)\.db$", filename_in)
if tagname :
    tagname  = tagname .group(1)
    log.info(f"tag = {tagname}")

In [ ]:
log.info("Configuration of opsim {opsim_path} done!")

## 3) Read Opsim Db

In [ ]:
sql_engine = op.OpSimSurvey._get_sql_engine(opsim_path)

In [ ]:
df = op.OpSimSurvey._get_df_from_sql(sql_engine)

In [ ]:
log.info(f"columns : {list(df.columns)}")

## 4) Analyse Setup

In [ ]:
BANDS_ORDER = ["ugrizy"]

In [ ]:
all_df = [df[df["filter"] == b] for b in BANDS_ORDER]

### 4.1)  Split datafrape per band

In [ ]:
df["filter"] = df["filter"].str.extract(r"([a-zA-Z]+)")

In [ ]:
df["filter"]

In [ ]:
df

## 5) Plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.time import Time

In [ ]:
import ipywidgets

%matplotlib widget

import ipywidgets as widgets

widgets.IntSlider()

print("matplotlib:", mpl.__version__)
print("backend:", mpl.get_backend())
print("ipywidgets:", ipywidgets.__version__)

In [ ]:
# Standard Rubin/LSST filter colors
FILTER_COLORS = {
    "u": "#56b4e9",
    "g": "#008060",
    "r": "#ff4000",
    "i": "#850000",
    "z": "#6600cc",
    "y": "#000000",
}
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]

### 5.1) Helpers

In [ ]:
# ── savefig: PDF + PNG ───────────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    #fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

In [ ]:
def get_year_ticks_bad(mjd_min, mjd_max):
    """
    Compute the MJD of January 1st for every year covered by
    [mjd_min, mjd_max], used for axis ticks and axvlines.

    .datetime force une conversion vers calendrier civil
    → appelle ERFA d2dtf
    → déclenche "dubious year"
    
    """
    t_min = Time(mjd_min, format="mjd").datetime
    t_max = Time(mjd_max, format="mjd").datetime

    year_mjds = []
    year_labels = []
    for year in range(t_min.year, t_max.year + 2):
        t_jan1 = Time(f"{year}-01-01T00:00:00", format="isot", scale="utc")
        if mjd_min <= t_jan1.mjd <= mjd_max:
            year_mjds.append(t_jan1.mjd)
            year_labels.append(f"{year}-01-01")
    return np.array(year_mjds), year_labels

In [ ]:
def get_year_ticks(mjd_min, mjd_max):
    """
    Compute the MJD of January 1st for every year covered by
    [mjd_min, mjd_max], used for axis ticks and axvlines.

    
    """
    t_min = Time(mjd_min, format="mjd", scale="utc")
    t_max = Time(mjd_max, format="mjd", scale="utc")

    # Extraire les années via format string (pas datetime)
    year_min = int(t_min.strftime("%Y"))
    year_max = int(t_max.strftime("%Y"))

    year_mjds = []
    year_labels = []

    for year in range(year_min, year_max + 2):
        t_jan1 = Time(f"{year}-01-01T00:00:00", format="isot", scale="utc")
        if mjd_min <= t_jan1.mjd <= mjd_max:
            year_mjds.append(t_jan1.mjd)
            year_labels.append(f"{year}-01-01")

    return np.array(year_mjds), year_labels

### 5.2) Categorical columns: stacked horizontal bar plots by band

In [ ]:
def plot_category_stacked_barh(df, column, filter_column="filter", max_categories=30, figsize=(10, 8),simuname=tagname):
    """
    Plot a horizontal bar chart of value counts for a categorical column,
    with each bar stacked by band (in FILTER_ORDER), using the standard
    Rubin/LSST filter colors.
    """
    counts = pd.crosstab(df[column], df[filter_column])
    bands = [b for b in FILTER_ORDER if b in counts.columns]
    counts = counts[bands]

    # Keep only the most populated categories for readability
    totals = counts.sum(axis=1).sort_values(ascending=False)
    counts = counts.loc[totals.index[:max_categories]]
    # Reverse so the largest category appears at the top of the plot
    counts = counts.iloc[::-1]

    fig, ax = plt.subplots(figsize=figsize)
    left = np.zeros(len(counts))
    for band in bands:
        ax.barh(
            counts.index.astype(str),
            counts[band].values,
            left=left,
            color=FILTER_COLORS[band],
            label=band,
        )
        left = left + counts[band].values

    ax.set_xlabel("Number of observations")
    ax.set_ylabel(column)
    ax.set_title(f"{column} by band ({simuname})")
    ax.legend(title="band", loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
    fig.tight_layout()
    return fig, ax

In [ ]:
# Columns not plotted: the ones already used to define the bands
CATEGORY_SKIP_COLUMNS = {"filter", "band"}

for column in df.columns:
    if column in CATEGORY_SKIP_COLUMNS:
        continue
    if pd.api.types.is_numeric_dtype(df[column]):
        continue
    fig, ax = plot_category_stacked_barh(df, column)
    figname = f"{tagname}_{column}"
    savefig(fig, figname, dpi=150)
    plt.show()

### 5.3) Numerical Columns: Visualize each column vs MJD, colored by band

In [ ]:
def plot_column_vs_mjd(
    df, column, mjd_column="observationStartMJD", filter_column="filter", figsize=(12, 5), simuname=tagname):
    """
    Plot a dataframe column versus MJD, with points colored by
    Rubin/LSST band, a secondary top axis showing readable dates
    (YYYY-MM-DD), and vertical dashed lines marking the start of each year.
    """
    fig, ax = plt.subplots(figsize=figsize)

    mjd = df[mjd_column].values
    values = df[column]

    # Scatter per band so each band gets its own color and legend entry
    for band in FILTER_ORDER:
        mask = df[filter_column] == band
        if mask.sum() == 0:
            continue
        ax.scatter(
            mjd[mask.values],
            values[mask].values,
            s=3,
            alpha=0.5,
            color=FILTER_COLORS[band],
            label=band,
        )

    ax.set_xlabel("MJD")
    ax.set_ylabel(column)
    ax.set_title(f"{column} vs MJD ({simuname})")
    ax.legend(markerscale=5, title="band", loc="center left", fontsize=8, bbox_to_anchor=(1, 0.5))

    # Vertical lines at the start of each year
    mjd_min, mjd_max = np.nanmin(mjd), np.nanmax(mjd)
    year_mjds, year_labels = get_year_ticks(mjd_min, mjd_max)
    for ymjd in year_mjds:
        ax.axvline(ymjd, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

    # Secondary top axis with readable dates
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(year_mjds)
    ax_top.set_xticklabels(year_labels, rotation=45, ha="left")
    ax_top.set_xlabel("Date")

    fig.tight_layout()
    return fig, ax

In [ ]:
# Columns not plotted: the MJD itself, and non-numeric / categorical columns
SKIP_COLUMNS = {"observationStartMJD", "band", "filter"}

for column in df.columns:
    if column in SKIP_COLUMNS:
        continue
    if not pd.api.types.is_numeric_dtype(df[column]):
        continue
    fig, ax = plot_column_vs_mjd(df, column)
    # too many points saturate the figure
    #figname = f"{tagname}_{column}"
    #print(figname)
    #savefig(fig, figname, dpi=300)
    plt.show()